# 02. Inference three ways

Estimating the ATE is only half the job; the other half is saying **how sure** we are. The library offers three routes, with different interpretations:

- **Randomization** (`RandomizationTest`): tests Fisher's strong null hypothesis.
- **Neyman** (`NeymanCI`): finite-population variance.
- **Bootstrap** (`BootstrapCI`): superpopulation variance.

We run all three on the **same** experiment and compare.

## The experiment

Same pattern as notebook 01: potential outcomes with a true effect of `0.5`, complete randomization, observed outcome.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans

rng = np.random.default_rng(42)
n = 200
y0 = rng.normal(size=n)
tau = 0.5
y1 = y0 + tau

df = pd.DataFrame({"x": rng.normal(size=n)})
design = CRD(p=0.5, seed=42)
assignment = design.randomize(df)
t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=42
)

## 1. Randomization (Fisher's sharp null)

Permutes the treatment thousands of times under the null of "no effect for any unit". It yields a **p-value**, but no standard error or confidence interval.

In [ ]:
from skxperiments.inference import RandomizationTest

res_rand = RandomizationTest(
    estimator=DifferenceInMeans(outcome_col="y"), n_permutations=500, seed=0
).fit(assignment).estimate()
print(f"Randomization: ATE={res_rand.ate:.3f}, p={res_rand.p_value:.4f}")

## 2. Neyman (finite population)

Conservative variance of the ATE for **these** units. It yields a standard error, a confidence interval, and a Wald p-value.

In [ ]:
from skxperiments.inference import NeymanCI

res_ney = NeymanCI(
    estimator=DifferenceInMeans(outcome_col="y")
).fit(assignment).estimate()
print(
    f"Neyman: ATE={res_ney.ate:.3f}, SE={res_ney.se:.3f}, "
    f"CI=({res_ney.ci[0]:.3f}, {res_ney.ci[1]:.3f}), p={res_ney.p_value:.4f}"
)

## 3. Bootstrap (superpopulation)

Resamples units within each arm to approximate the sampling distribution, treating them as a sample from a larger population.

In [ ]:
from skxperiments.inference import BootstrapCI

res_boot = BootstrapCI(
    estimator=DifferenceInMeans(outcome_col="y"), n_resamples=1000, seed=0
).fit(assignment).estimate()
print(
    f"Bootstrap: ATE={res_boot.ate:.3f}, SE={res_boot.se:.3f}, "
    f"CI=({res_boot.ci[0]:.3f}, {res_boot.ci[1]:.3f}), p={res_boot.p_value:.4f}"
)

## Comparing

All three agree on the ATE (it is the same estimator); what changes is the uncertainty measure. Randomization does not produce a SE/CI natively (hence `None` in the table).

In [ ]:
pd.DataFrame(
    [
        {"method": "Randomization", "ATE": res_rand.ate, "SE": res_rand.se, "p": res_rand.p_value},
        {"method": "Neyman", "ATE": res_ney.ate, "SE": res_ney.se, "p": res_ney.p_value},
        {"method": "Bootstrap", "ATE": res_boot.ate, "SE": res_boot.se, "p": res_boot.p_value},
    ]
)

## When to use each

- **Randomization**: when you want a p-value that depends only on the design, with no distributional assumptions. Tests the sharp null.
- **Neyman**: when the question is about **these** units (finite population) and you want a fast, conservative CI.
- **Bootstrap**: when the units are a sample from a larger population (superpopulation) and the question is about that population.

**Next:** `03. Reducing variance` shows how covariates (Lin and CUPED) make the CI narrower.